In [8]:
import os
# 锁死 BLAS 单线程，防止 joblib worker 内部多线程导致 144 线程抢 12 核 / 内存爆炸。
# 这几行必须在 import numpy 之前。
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")  # macOS Apple Accelerate / vecLib
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from pipelines import estimate_regression_models_noLASSO
from joblib import Parallel, delayed
from tqdm import tqdm


In [9]:
def compute_errnorm_cauchy(i, p, n, nn, beta_0, w_0, sigma, sigma1, psi_delta=1.35, psi_eta=0.1, tau_range=None):
    np.random.seed(i)  # 为每个并行工作设置独立的种子

    # --- 数据生成 ---
    uniform_values1 = np.random.uniform(0, np.sqrt(3), n)
    X = np.random.normal(size=(n, p)) * uniform_values1[:, np.newaxis]
    Y = X @ beta_0 + np.random.standard_cauchy(size=n) * sigma

    # 源任务
    uniform_values2 = np.random.uniform(0, np.sqrt(3), nn)
    X1 = np.random.normal(size=(nn, p)) * uniform_values2[:, np.newaxis]
    Y1 = X1 @ w_0 + np.random.standard_cauchy(size=nn) * sigma1

    #####################
    if tau_range is None:
        tau_range = np.logspace(-4, 1, 15)  # 更细致的 tau 范围

    # 调用函数
    estimated_models = estimate_regression_models_noLASSO(
        X, Y,
        X1, Y1,
        delta_param=psi_delta,
        eta_param=psi_eta,
        tau_range=tau_range
    )

    beta_sr  = estimated_models["single_robust_ridge"]["betahat"]
    beta_tr  = estimated_models["transfer_robust_ridge"]["betahat"]
    beta_ada = estimated_models["adaptive"]["betahat"]
    beta_pr  = estimated_models["pooled_robust_ridge"]["betahat"]

    errnorm_sr  = np.sum((beta_sr  - beta_0) ** 2) / np.sum(beta_0 ** 2)
    errnorm_tr  = np.sum((beta_tr  - beta_0) ** 2) / np.sum(beta_0 ** 2)
    errnorm_ada = np.sum((beta_ada - beta_0) ** 2) / np.sum(beta_0 ** 2)
    errnorm_pr  = np.sum((beta_pr  - beta_0) ** 2) / np.sum(beta_0 ** 2)

    # 提取最优 tau 值
    optimal_tau_sr = estimated_models["single_robust_ridge"]["optimal_tau"]
    optimal_tau_source_tr = estimated_models["transfer_robust_ridge"]["optimal_tau_source"]
    optimal_tau_target_diff_tr = estimated_models["transfer_robust_ridge"]["optimal_tau_target_diff"]
    optimal_tau_pr = estimated_models["pooled_robust_ridge"]["optimal_tau"]

    return (errnorm_sr, errnorm_tr, errnorm_ada, errnorm_pr,
            optimal_tau_sr, optimal_tau_source_tr, optimal_tau_target_diff_tr, optimal_tau_pr)



In [10]:
def run_simulation_cv(p, n, K, dd, n_jobs=-1, tau_range=None):
    """
    运行完整的模拟研究，使用CV优化tau参数。

    Args:
        p (int): 特征维度
        n (int): 目标任务样本量
        K (int): 模拟重复次数
        dd (float): 控制delta_0范数的系数
        n_jobs (int): 并行使用的CPU核心数
        tau_range (array, optional): 用于CV的tau候选值范围

    Returns:
        tuple: (mean_errnorm, std_errnorm, errnorm_df, ridge_tau_stats)
    """
    # 简化输出，只显示进度条
    # print(f"Starting CV simulation with p={p}, n={n}, K={K}, dd={dd}")

    # --- 在函数内部定义固定的模拟参数 ---
    nn = n * 2  # 源任务样本量
    sigma = 1  # 目标任务噪声标准差
    sigma1 = 2  # 源任务噪声标准差
    psi_delta = 1.35  # psi 函数参数 delta
    psi_eta = 0.1  # psi 函数参数 eta
    kappa = p // n  # 维度样本比

    if tau_range is None:
        tau_range = np.logspace(-3, 3, 10)  # 默认的tau值范围

    # --- 生成固定的真实系数 (在所有 K 次运行中保持不变) ---
    rng = np.random.RandomState(1)  # 使用固定的种子以保证 beta_0, w_0 可复现
    beta_0 = rng.uniform(size=p)
    beta_0 /= np.linalg.norm(beta_0, 2)

    # delta_0 = rng.uniform(size=p)
    # delta_0 /= np.linalg.norm(delta_0, 2) * dd
    delta_0 = np.ones(p) * dd / np.sqrt(p)  # 固定的 delta_0

    w_0 = beta_0 - delta_0

    # --- 使用 joblib 进行并行计算 ---
    # 简化输出，只显示进度条
    # print(f"Running {K} simulations in parallel using {n_jobs if n_jobs > 0 else os.cpu_count()} cores...")
    results_list = Parallel(n_jobs=n_jobs)(
        delayed(compute_errnorm_cauchy)(
            i, p, n, nn, beta_0, w_0, sigma, sigma1, psi_delta, psi_eta, tau_range
        )
        for i in tqdm(range(K), desc=f"dd={dd:.3f}")
    )


    # --- 结果处理 ---
    # results_list 中的每个元素是: (err_sr, err_tr, err_pr, tau_sr, tau_tr_s, tau_tr_td, tau_pr)
    errnorm_values = np.array([r[0:4] for r in results_list])  # K x 4 array (incl. Adaptive)
    tau_values = np.array([r[4:8] for r in results_list])      # K x 4 array

    mean_errnorm = np.nanmean(errnorm_values, axis=0)
    std_errnorm = np.nanstd(errnorm_values, axis=0)


    errnorm_df = pd.DataFrame(errnorm_values, columns=['Single RR', 'Trans RR', 'Trans-RR-Ada', 'Pooled RR'])

    tau_columns = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    all_taus_df = pd.DataFrame(tau_values, columns=tau_columns)

    return mean_errnorm, std_errnorm, errnorm_df, all_taus_df




In [11]:
p_val = 400
n_val = 400
K_val = 1000  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = 8  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(0, 1, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(np.e, np.arange(-2.0, 1.5, 0.5))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:

    mean_err, std_err, results_df, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 计算每个 tau 列的众数和频次
    tau_stats_for_dd = {}
    tau_columns_for_stats = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    for col_name in tau_columns_for_stats:
        if col_name in all_t_df:
            counts = all_t_df[col_name].value_counts()
            if not counts.empty:
                most_frequent_tau = counts.index[0]
                frequency = counts.iloc[0]
                tau_stats_for_dd[col_name] = (most_frequent_tau, frequency)
            else:
                tau_stats_for_dd[col_name] = (np.nan, 0) # 处理空 Series 的情况
        else:
            tau_stats_for_dd[col_name] = (np.nan, 0) # 如果列不存在

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'all_taus_df': all_t_df,
            'tau_stats': tau_stats_for_dd # 新增 tau 统计信息
        }

# 汇总错误结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage:
        for i, method in enumerate(error_method_names):
            mean = all_results_storage[dd_val]['mean_err'][i]
            std = all_results_storage[dd_val]['std_err'][i]
            results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
        print(results_line)
    else:
        print(f"{dd_val:<10.3f} No results found.") # 以防万一


# 汇总 Tau 统计结果
print("\n\n" + "=" * 130)
print("Summary of Most Frequent Tau Values and Their Counts Across Different dd Values".center(130))
print("=" * 130)

# tau_columns_for_stats 已在上面定义
# ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
header_tau = f"{'dd':<10}"
for tau_method_name in tau_columns_for_stats:
    header_tau += f"{tau_method_name + ' (Freq)':<30}" # 调整宽度
print(header_tau)
print("-" * 130)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage and 'tau_stats' in all_results_storage[dd_val]:
        stats = all_results_storage[dd_val]['tau_stats']
        for tau_method_name in tau_columns_for_stats:
            if tau_method_name in stats:
                tau_val, freq = stats[tau_method_name]
                tau_results_line += f"{tau_val:.4f} ({freq}/{K_val})".ljust(30) # 显示频次和总次数
            else:
                tau_results_line += "N/A".ljust(30)
        print(tau_results_line)
    else:
        print(f"{dd_val:<10.3f} No tau stats found.")

print("\nSimulation complete.")

dd=2.718: 100%|██████████| 1000/1000 [07:55<00:00,  2.10it/s]




                   Summary of CV Results Across Different dd Values - Mean Error (Std Dev)                    
dd        Single RR                     Trans RR                      Pooled RR                     
--------------------------------------------------------------------------------------------------------------
0.135     0.8644 (0.0366)               0.7846 (0.0425)               0.8042 (0.0412)               
0.223     0.8644 (0.0366)               0.7937 (0.0422)               0.8198 (0.0394)               
0.368     0.8644 (0.0366)               0.8094 (0.0420)               0.8448 (0.0351)               
0.607     0.8644 (0.0366)               0.8376 (0.0422)               0.8826 (0.0272)               
1.000     0.8644 (0.0366)               0.8917 (0.0455)               0.9360 (0.0205)               
1.649     0.8644 (0.0366)               1.0280 (0.0994)               1.0129 (0.0273)               
2.718     0.8644 (0.0366)               1.6184 (0.1693)              

In [12]:
import json

filename = f"res/1cauchy_cv_results_p{p_val}_simu{K_val}.json"

converted_results = {}
for dd_val in all_results_storage:
    converted_results[str(dd_val)] = {
        'mean_err': all_results_storage[dd_val]['mean_err'].tolist(),
        'std_err': all_results_storage[dd_val]['std_err'].tolist(),
        'results_df': all_results_storage[dd_val]['results_df'].to_dict()
    }

with open(filename, 'w') as f:
    json.dump(converted_results, f, indent=4)

print(f"结果已保存至: {filename}")

结果已保存至: res/1cauchy_cv_results_p400_simu1000.json


In [4]:
p_val = 400
n_val = 400
K_val = 500  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = 8  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(0, 1, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(np.e, np.arange(-0.5, 2.5, 0.5))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:

    mean_err, std_err, results_df, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 计算每个 tau 列的众数和频次
    tau_stats_for_dd = {}
    tau_columns_for_stats = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    for col_name in tau_columns_for_stats:
        if col_name in all_t_df:
            counts = all_t_df[col_name].value_counts()
            if not counts.empty:
                most_frequent_tau = counts.index[0]
                frequency = counts.iloc[0]
                tau_stats_for_dd[col_name] = (most_frequent_tau, frequency)
            else:
                tau_stats_for_dd[col_name] = (np.nan, 0) # 处理空 Series 的情况
        else:
            tau_stats_for_dd[col_name] = (np.nan, 0) # 如果列不存在

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'all_taus_df': all_t_df,
            'tau_stats': tau_stats_for_dd # 新增 tau 统计信息
        }

# 汇总错误结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage:
        for i, method in enumerate(error_method_names):
            mean = all_results_storage[dd_val]['mean_err'][i]
            std = all_results_storage[dd_val]['std_err'][i]
            results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
        print(results_line)
    else:
        print(f"{dd_val:<10.3f} No results found.") # 以防万一


# 汇总 Tau 统计结果
print("\n\n" + "=" * 130)
print("Summary of Most Frequent Tau Values and Their Counts Across Different dd Values".center(130))
print("=" * 130)

# tau_columns_for_stats 已在上面定义
# ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
header_tau = f"{'dd':<10}"
for tau_method_name in tau_columns_for_stats:
    header_tau += f"{tau_method_name + ' (Freq)':<30}" # 调整宽度
print(header_tau)
print("-" * 130)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage and 'tau_stats' in all_results_storage[dd_val]:
        stats = all_results_storage[dd_val]['tau_stats']
        for tau_method_name in tau_columns_for_stats:
            if tau_method_name in stats:
                tau_val, freq = stats[tau_method_name]
                tau_results_line += f"{tau_val:.4f} ({freq}/{K_val})".ljust(30) # 显示频次和总次数
            else:
                tau_results_line += "N/A".ljust(30)
        print(tau_results_line)
    else:
        print(f"{dd_val:<10.3f} No tau stats found.")

print("\nSimulation complete.")

dd=7.389: 100%|██████████| 500/500 [03:51<00:00,  2.16it/s]




                   Summary of CV Results Across Different dd Values - Mean Error (Std Dev)                    
dd        Single RR                     Trans RR                      Pooled RR                     
--------------------------------------------------------------------------------------------------------------
0.607     0.8666 (0.0390)               1.0417 (0.1196)               0.9966 (0.0363)               
1.000     0.8666 (0.0390)               0.8838 (0.0500)               0.9197 (0.0239)               
1.649     0.8666 (0.0390)               0.8317 (0.0431)               0.8692 (0.0308)               
2.718     0.8666 (0.0390)               0.8059 (0.0416)               0.8362 (0.0369)               
4.482     0.8666 (0.0390)               0.7920 (0.0419)               0.8150 (0.0399)               
7.389     0.8666 (0.0390)               0.7844 (0.0422)               0.8013 (0.0414)               


                         Summary of Most Frequent Tau Values and Th

In [7]:
import json

filename = f"res/cauchy_cv_results_p{p_val}_simu{K_val}.json"

converted_results = {}
for dd_val in all_results_storage:
    converted_results[str(dd_val)] = {
        'mean_err': all_results_storage[dd_val]['mean_err'].tolist(),
        'std_err': all_results_storage[dd_val]['std_err'].tolist(),
        'results_df': all_results_storage[dd_val]['results_df'].to_dict()
    }

with open(filename, 'w') as f:
    json.dump(converted_results, f, indent=4)

print(f"结果已保存至: {filename}")

结果已保存至: res/cauchy_cv_results_p400_simu500.json


In [5]:
p_val = 400
n_val = 400
K_val = 50  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = 8  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(0, 2, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(2.0, np.arange(-2, 3.5, 0.5))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:

    mean_err, std_err, results_df, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 计算每个 tau 列的众数和频次
    tau_stats_for_dd = {}
    tau_columns_for_stats = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    for col_name in tau_columns_for_stats:
        if col_name in all_t_df:
            counts = all_t_df[col_name].value_counts()
            if not counts.empty:
                most_frequent_tau = counts.index[0]
                frequency = counts.iloc[0]
                tau_stats_for_dd[col_name] = (most_frequent_tau, frequency)
            else:
                tau_stats_for_dd[col_name] = (np.nan, 0) # 处理空 Series 的情况
        else:
            tau_stats_for_dd[col_name] = (np.nan, 0) # 如果列不存在

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'all_taus_df': all_t_df,
            'tau_stats': tau_stats_for_dd # 新增 tau 统计信息
        }

# 汇总错误结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage:
        for i, method in enumerate(error_method_names):
            mean = all_results_storage[dd_val]['mean_err'][i]
            std = all_results_storage[dd_val]['std_err'][i]
            results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
        print(results_line)
    else:
        print(f"{dd_val:<10.3f} No results found.") # 以防万一


# 汇总 Tau 统计结果
print("\n\n" + "=" * 130)
print("Summary of Most Frequent Tau Values and Their Counts Across Different dd Values".center(130))
print("=" * 130)

# tau_columns_for_stats 已在上面定义
# ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
header_tau = f"{'dd':<10}"
for tau_method_name in tau_columns_for_stats:
    header_tau += f"{tau_method_name + ' (Freq)':<30}" # 调整宽度
print(header_tau)
print("-" * 130)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage and 'tau_stats' in all_results_storage[dd_val]:
        stats = all_results_storage[dd_val]['tau_stats']
        for tau_method_name in tau_columns_for_stats:
            if tau_method_name in stats:
                tau_val, freq = stats[tau_method_name]
                tau_results_line += f"{tau_val:.4f} ({freq}/{K_val})".ljust(30) # 显示频次和总次数
            else:
                tau_results_line += "N/A".ljust(30)
        print(tau_results_line)
    else:
        print(f"{dd_val:<10.3f} No tau stats found.")

print("\nSimulation complete.")

dd=8.000: 100%|██████████| 50/50 [00:15<00:00,  3.18it/s]




                   Summary of CV Results Across Different dd Values - Mean Error (Std Dev)                    
dd        Single RR                     Trans RR                      Pooled RR                     
--------------------------------------------------------------------------------------------------------------
0.250     0.8847 (0.0372)               1.8807 (0.1219)               1.3700 (0.1838)               
0.354     0.8847 (0.0372)               1.5929 (0.1271)               1.1629 (0.1273)               
0.500     0.8847 (0.0372)               1.1899 (0.1634)               1.0368 (0.0562)               
0.707     0.8847 (0.0372)               0.9635 (0.0826)               0.9716 (0.0207)               
1.000     0.8847 (0.0372)               0.8901 (0.0451)               0.9276 (0.0284)               
1.414     0.8847 (0.0372)               0.8649 (0.0401)               0.8916 (0.0395)               
2.000     0.8847 (0.0372)               0.8504 (0.0407)              

In [7]:
p_val = 200
n_val = 200
K_val = 50  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = 8  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(0, 1, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(2.0, np.arange(-2, 3.5, 0.5))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:

    mean_err, std_err, results_df, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 计算每个 tau 列的众数和频次
    tau_stats_for_dd = {}
    tau_columns_for_stats = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    for col_name in tau_columns_for_stats:
        if col_name in all_t_df:
            counts = all_t_df[col_name].value_counts()
            if not counts.empty:
                most_frequent_tau = counts.index[0]
                frequency = counts.iloc[0]
                tau_stats_for_dd[col_name] = (most_frequent_tau, frequency)
            else:
                tau_stats_for_dd[col_name] = (np.nan, 0) # 处理空 Series 的情况
        else:
            tau_stats_for_dd[col_name] = (np.nan, 0) # 如果列不存在

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'all_taus_df': all_t_df,
            'tau_stats': tau_stats_for_dd # 新增 tau 统计信息
        }

# 汇总错误结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage:
        for i, method in enumerate(error_method_names):
            mean = all_results_storage[dd_val]['mean_err'][i]
            std = all_results_storage[dd_val]['std_err'][i]
            results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
        print(results_line)
    else:
        print(f"{dd_val:<10.3f} No results found.") # 以防万一


# 汇总 Tau 统计结果
print("\n\n" + "=" * 130)
print("Summary of Most Frequent Tau Values and Their Counts Across Different dd Values".center(130))
print("=" * 130)

# tau_columns_for_stats 已在上面定义
# ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
header_tau = f"{'dd':<10}"
for tau_method_name in tau_columns_for_stats:
    header_tau += f"{tau_method_name + ' (Freq)':<30}" # 调整宽度
print(header_tau)
print("-" * 130)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage and 'tau_stats' in all_results_storage[dd_val]:
        stats = all_results_storage[dd_val]['tau_stats']
        for tau_method_name in tau_columns_for_stats:
            if tau_method_name in stats:
                tau_val, freq = stats[tau_method_name]
                tau_results_line += f"{tau_val:.4f} ({freq}/{K_val})".ljust(30) # 显示频次和总次数
            else:
                tau_results_line += "N/A".ljust(30)
        print(tau_results_line)
    else:
        print(f"{dd_val:<10.3f} No tau stats found.")

print("\nSimulation complete.")

dd=8.000: 100%|██████████| 50/50 [00:06<00:00,  8.20it/s]




                   Summary of CV Results Across Different dd Values - Mean Error (Std Dev)                    
dd        Single RR                     Trans RR                      Pooled RR                     
--------------------------------------------------------------------------------------------------------------
0.250     0.8725 (0.0508)               2.0171 (0.1668)               1.5911 (0.1599)               
0.354     0.8725 (0.0508)               1.7058 (0.1312)               1.3688 (0.1465)               
0.500     0.8725 (0.0508)               1.3633 (0.1244)               1.1339 (0.0921)               
0.707     0.8725 (0.0508)               1.0208 (0.0974)               0.9794 (0.0460)               
1.000     0.8725 (0.0508)               0.8705 (0.0611)               0.8944 (0.0419)               
1.414     0.8725 (0.0508)               0.8072 (0.0539)               0.8321 (0.0504)               
2.000     0.8725 (0.0508)               0.7695 (0.0535)              

In [12]:
p_val = 200
n_val = 200
K_val = 50  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = 8  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(-2, 1, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(2.0, np.arange(-2, 3.5, 0.5))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:
    # 简化输出，移除分隔线
    # print(f"\n\n{'=' * 50}")
    # print(f"Running CV simulation with dd = {dd_val}")
    # print(f"{'=' * 50}")

    # 运行模拟
    mean_err, std_err, results_df, mean_t, std_t, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'mean_taus': mean_t,
            'std_taus': std_t,
            'all_taus_df': all_t_df
        }

# 汇总结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    for i, method in enumerate(error_method_names):
        mean = all_results_storage[dd_val]['mean_err'][i]
        std = all_results_storage[dd_val]['std_err'][i]
        results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
    print(results_line)

# 汇总结果 - Tau 值
tau_method_names = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Optimal Tau (Std Dev)".center(110))
print("=" * 110)

header_tau = f"{'dd':<10}"
for method in tau_method_names:
    header_tau += f"{method:<25}" # Tau值通常不需要那么宽的列
print(header_tau)
print("-" * 110)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    current_mean_taus = all_results_storage[dd_val]['mean_taus']
    current_std_taus = all_results_storage[dd_val]['std_taus']
    for i, method in enumerate(tau_method_names):
        mean = current_mean_taus[i]
        std = current_std_taus[i]
        tau_results_line += f"{mean:.4f} ({std:.4f})".ljust(25)
    print(tau_results_line)
print("-" * 110)

dd=8.000: 100%|████████████████████████████████████████████████████████████████████████| 50/50 [00:37<00:00,  1.33it/s]




                   Summary of CV Results Across Different dd Values - Mean Error (Std Dev)                    
dd        Single RR                     Trans RR                      Pooled RR                     
--------------------------------------------------------------------------------------------------------------
0.250     0.8738 (0.0502)               6.1048 (1.1772)               2.4981 (0.9537)               
0.354     0.8738 (0.0502)               2.9646 (0.5403)               1.5275 (0.3645)               
0.500     0.8738 (0.0502)               1.5575 (0.2750)               1.1412 (0.1120)               
0.707     0.8738 (0.0502)               1.0279 (0.1172)               0.9798 (0.0479)               
1.000     0.8738 (0.0502)               0.8695 (0.0648)               0.8945 (0.0409)               
1.414     0.8738 (0.0502)               0.8066 (0.0545)               0.8329 (0.0507)               
2.000     0.8738 (0.0502)               0.7692 (0.0540)              

In [13]:
for dd_val, results_dict_for_dd in all_results_storage.items():
    current_all_taus_df = results_dict_for_dd['all_taus_df']
    
    first_method_output_for_this_dd = True
    
    for tau_method_name in current_all_taus_df.columns:
        # 获取当前tau方法对应的Series
        tau_values_series = current_all_taus_df[tau_method_name]

        # 计算该列中每个tau值的出现频率
        # value_counts() 默认按频率降序排序，并且不计算NaN值
        frequencies = tau_values_series.value_counts()

        # 准备dd值的显示字符串
        dd_display_string = f"{dd_val:<10.3f}" if first_method_output_for_this_dd else " " * 10

        # 获取最高的频率计数
        max_count = frequencies.iloc[0]
        # 找到所有具有此最高频率的tau值 (处理并列情况)
        most_frequent_tau_values = frequencies[frequencies == max_count]

        # 格式化这些tau值（它们是来自您优化时使用的tau_range的实际值）
        tau_values_str_list = [f"{tau_val:.4f}" for tau_val in most_frequent_tau_values.index]
        most_frequent_taus_display = ", ".join(tau_values_str_list)
        count_display = str(max_count)
        
        print(f"{dd_display_string}"
              f"{tau_method_name:<25}"
              f"{most_frequent_taus_display:<45}"
              f"{count_display:<10}")
        first_method_output_for_this_dd = False # 后续行不再打印dd值
        
        # 在每个dd_val的方法列表结束后，可以加一个子分隔符（如果为该dd处理了数据且不是最后一个dd）
        if not first_method_output_for_this_dd: # 意味着至少有一个tau method被处理了
            if dd_val != list(all_results_storage.keys())[-1]:
                print(" " * 10 + "-" * 100) # 子分隔符

print("=" * 110) # 表格结束的总分隔符

0.250     Tau SR                   3.0000                                       33        
          ----------------------------------------------------------------------------------------------------
          Tau TR (Source)          0.1602                                       27        
          ----------------------------------------------------------------------------------------------------
          Tau TR (Target Diff)     0.4807                                       20        
          ----------------------------------------------------------------------------------------------------
          Tau PR                   0.6934, 0.3333                               12        
          ----------------------------------------------------------------------------------------------------
0.354     Tau SR                   3.0000                                       33        
          -------------------------------------------------------------------------------------------

In [10]:
print("\n\n" + "=" * 110) # 表格的总宽度
print("Most Frequent Optimal Raw Tau Values and Their Counts (from all_taus_df)".center(110))
print("=" * 110)

# 定义表头
header_line = f"{'dd':<10}{'Tau Method':<25}{'Most Frequent Tau(s)':<45}{'Count':<10}"
print(header_line)
print("-" * 110)

# 遍历 all_results_storage 中的每一个 dd_val
for dd_val, results_dict_for_dd in all_results_storage.items():

    # 检查 'all_taus_df' 是否存在于当前 dd_val 的结果字典中
    if 'all_taus_df' not in results_dict_for_dd:
        print(f"{dd_val:<10.3f}{'\'all_taus_df\' key not found in results for this dd.':<100}")
        if dd_val != list(all_results_storage.keys())[-1]: # 不是最后一个dd，则打印分隔线
             print(" " * 10 + "-" * 100)
        continue

    current_all_taus_df = results_dict_for_dd['all_taus_df']

    # 确保它是一个DataFrame并且不为空
    if not isinstance(current_all_taus_df, pd.DataFrame) or current_all_taus_df.empty:
        print(f"{dd_val:<10.3f}{'\'all_taus_df\' is not a valid or non-empty DataFrame.':<100}")
        if dd_val != list(all_results_storage.keys())[-1]:
             print(" " * 10 + "-" * 100)
        continue

    first_method_output_for_this_dd = True # 控制dd值的打印，使其只在每个dd块的第一行出现

    # 遍历 DataFrame 中的每一列 (即每一种tau的计算方法)
    for tau_method_name in current_all_taus_df.columns:
        # 获取当前tau方法对应的Series
        tau_values_series = current_all_taus_df[tau_method_name]

        # 计算该列中每个tau值的出现频率
        # value_counts() 默认按频率降序排序，并且不计算NaN值
        frequencies = tau_values_series.value_counts()

        # 准备dd值的显示字符串
        dd_display_string = f"{dd_val:<10.3f}" if first_method_output_for_this_dd else " " * 10

        if frequencies.empty:
            # 如果该列数据为空或全为NaN，则frequencies会为空
            most_frequent_taus_display = "N/A (No valid data)"
            count_display = "N/A"
        else:
            # 获取最高的频率计数
            max_count = frequencies.iloc[0]
            # 找到所有具有此最高频率的tau值 (处理并列情况)
            most_frequent_tau_values = frequencies[frequencies == max_count]

            # 格式化这些tau值（它们是来自您优化时使用的tau_range的实际值）
            tau_values_str_list = [f"{tau_val:.4f}" for tau_val in most_frequent_tau_values.index]
            most_frequent_taus_display = ", ".join(tau_values_str_list)
            count_display = str(max_count)

        # 打印当前 tau 方法的统计结果
        print(f"{dd_display_string}"
              f"{tau_method_name:<25}"
              f"{most_frequent_taus_display:<45}"
              f"{count_display:<10}")
        first_method_output_for_this_dd = False # 后续行不再打印dd值

    # 在每个dd_val的方法列表结束后，可以加一个子分隔符（如果为该dd处理了数据且不是最后一个dd）
    if not first_method_output_for_this_dd: # 意味着至少有一个tau method被处理了
        if dd_val != list(all_results_storage.keys())[-1]:
            print(" " * 10 + "-" * 100) # 子分隔符

print("=" * 110) # 表格结束的总分隔符

SyntaxError: f-string expression part cannot include a backslash (3730568504.py, line 15)

In [11]:
p_val = 400
n_val = 400
K_val = 50  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = 8  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(-1, 2, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(2.0, np.arange(-2, 3.5, 2))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:
    # 简化输出，移除分隔线
    # print(f"\n\n{'=' * 50}")
    # print(f"Running CV simulation with dd = {dd_val}")
    # print(f"{'=' * 50}")

    # 运行模拟
    mean_err, std_err, results_df, mean_t, std_t, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'mean_taus': mean_t,
            'std_taus': std_t,
            'all_taus_df': all_t_df
        }

# 汇总结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    for i, method in enumerate(error_method_names):
        mean = all_results_storage[dd_val]['mean_err'][i]
        std = all_results_storage[dd_val]['std_err'][i]
        results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
    print(results_line)

# 汇总结果 - Tau 值
tau_method_names = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Optimal Tau (Std Dev)".center(110))
print("=" * 110)

header_tau = f"{'dd':<10}"
for method in tau_method_names:
    header_tau += f"{method:<25}" # Tau值通常不需要那么宽的列
print(header_tau)
print("-" * 110)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    current_mean_taus = all_results_storage[dd_val]['mean_taus']
    current_std_taus = all_results_storage[dd_val]['std_taus']
    for i, method in enumerate(tau_method_names):
        mean = current_mean_taus[i]
        std = current_std_taus[i]
        tau_results_line += f"{mean:.4f} ({std:.4f})".ljust(25)
    print(tau_results_line)
print("-" * 110)

dd=0.250:  48%|██████████████████████████████████▌                                     | 24/50 [00:25<00:27,  1.06s/it]

KeyboardInterrupt: 

In [11]:
all_results_storage[4]

{'mean_err': array([5.27131736, 7.30727112, 1.64962899]),
 'std_err': array([16.21214943, 28.07894242,  1.54920411]),
 'results_df':     Single RR    Trans RR  Pooled RR
 0    0.907040    0.721489   0.684822
 1    0.921612    0.863663   1.629249
 2    0.895369    0.848137   0.841431
 3   28.174635   10.021154   4.430973
 4    0.942541    0.887721   0.792552
 5    8.920861    8.932173   4.853791
 6    0.917755    0.767775   0.927191
 7    4.336802    4.311576   2.432457
 8    0.936684    0.773023   0.937505
 9    0.848613    0.690306   0.647481
 10   0.927420    0.730710   0.927037
 11   1.896474    2.339994   0.752894
 12   0.930724    2.610369   0.928377
 13   4.695236    5.841758   3.369539
 14   0.915152    1.062095   0.935577
 15   0.933885    0.875180   0.931960
 16   1.751859    3.492045   6.032924
 17   1.975271    1.885907   0.927166
 18  93.652403   91.328590   0.929482
 19   0.937728    0.965866   2.006673
 20   0.845278    0.791380   0.924363
 21   1.673033    1.560020   1.9

In [11]:
p = 200
n = 200
K = 50  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = 8  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range = np.logspace(-1, 2, 10, base=3)
dd = 1

nn = n * 2  # 源任务样本量
sigma = 1  # 目标任务噪声标准差
sigma1 = 2  # 源任务噪声标准差
psi_delta = 1.35  # psi 函数参数 delta
psi_eta = 0.1  # psi 函数参数 eta
kappa = p // n  # 维度样本比

if tau_range is None:
    tau_range = np.logspace(-3, 3, 10)  # 默认的tau值范围

# --- 生成固定的真实系数 (在所有 K 次运行中保持不变) ---
rng = np.random.RandomState(1)  # 使用固定的种子以保证 beta_0, w_0 可复现
beta_0 = rng.uniform(size=p)
beta_0 /= np.linalg.norm(beta_0, 2)

delta_0 = rng.uniform(size=p)
delta_0 /= np.linalg.norm(delta_0, 2) * dd

w_0 = beta_0 - delta_0

result = compute_errnorm_cauchy(
            2, p, n, nn, beta_0, w_0, sigma, sigma1, psi_delta, psi_eta, tau_range
        )